In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\ThinkPad\Desktop\IKEA-main\retail_cleaned.csv",
    parse_dates=["invoice_date"],
    dtype={"invoice": "string", "stock_code": "string"},
)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1033036 non-null  string        
 1   stock_code    1033036 non-null  string        
 2   description   1028761 non-null  object        
 3   quantity      1033036 non-null  int64         
 4   invoice_date  1033036 non-null  datetime64[ns]
 5   price         1033036 non-null  float64       
 6   customer_id   797885 non-null   float64       
 7   country       1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(2), string(2)
memory usage: 63.1+ MB


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom


In [2]:
valid = df[(df["quantity"] > 0) & (df["customer_id"].notna())].copy()
valid["customer_id"] = valid["customer_id"].astype(int)
valid["revenue"] = valid["quantity"] * valid["price"]

snapshot_date = valid["invoice_date"].max().normalize() + pd.Timedelta(days=1)

rfm = valid.groupby("customer_id").agg(
    last_purchase=("invoice_date", "max"),
    frequency=("invoice", "nunique"),
    monetary=("revenue", "sum"),
).reset_index()

rfm["recency"] = (snapshot_date - rfm["last_purchase"].dt.normalize()).dt.days
rfm = rfm[["customer_id", "recency", "frequency", "monetary"]]

print("Snapshot date:", snapshot_date)
print("Baris valid  :", len(valid))
print("Jumlah customer:", len(rfm))
rfm.describe()

Snapshot date: 2011-12-10 00:00:00
Baris valid  : 779495
Jumlah customer: 5881


,customer_id,recency,frequency,monetary
count,5881.000000,5881.000000,5881.000000,5881.000000
mean,15314.674205,201.992858,6.287196,2954.396237
std,1715.429759,209.489651,13.012879,14437.322635
min,12346.000000,1.000000,1.000000,0.000000
25%,13833.000000,26.000000,1.000000,341.900000
50%,15313.000000,96.000000,3.000000,865.600000
75%,16797.000000,380.000000,7.000000,2247.720000
max,18287.000000,739.000000,398.000000,580987.040000


In [3]:
print("Customer dengan monetary <= 0:", (rfm["monetary"] <= 0).sum())

rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["rfm_score"] = (
    rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
)

# cek 1: tiap skor harus berisi sekitar 1.176 customer
print(rfm[["r_score", "f_score", "m_score"]].apply(lambda s: s.value_counts().sort_index()))

# cek 2: batas recency per skor R (skor 5 = recency terkecil)
print(rfm.groupby("r_score")["recency"].agg(["min", "max"]))

rfm.head()

Customer dengan monetary <= 0: 3
   r_score  f_score  m_score
1     1170     1177     1177
2     1175     1176     1176
3     1180     1176     1176
4     1140     1176     1176
5     1216     1176     1176
         min  max
r_score          
1        412  739
2        191  411
3         60  190
4         22   59
5          1   20


,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
0,12346,326,12,77556.46,2,5,5,255
1,12347,3,8,4921.53,5,4,5,545
2,12348,76,5,2019.40,3,4,4,344
3,12349,19,4,4428.69,5,3,5,535
4,12350,311,1,334.40,2,1,2,212


In [4]:
def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"          # baru beli, sering, nilai tinggi
    if r >= 3 and f >= 3 and m >= 3:
        return "Loyal"              # konsisten, nilai sedang-tinggi
    if r >= 4 and row["frequency"] == 1:
        return "New"                # baru, baru sekali beli
    if r <= 2 and f >= 3 and m >= 3:
        return "At Risk"            # dulu bernilai, kini recency turun
    if r >= 3:
        return "Needs Attention"    # recency sedang, frekuensi/nilai rendah
    return "Lost"                   # lama tidak aktif, nilai rendah

rfm["segment"] = rfm.apply(assign_segment, axis=1)

# churn: tidak ada pembelian dalam 90 hari terakhir (sesuai BRD)
rfm["churn"] = (rfm["recency"] > 90).astype(int)
rfm["churn_status"] = rfm["churn"].map({1: "Churned", 0: "Active"})

summary = rfm.groupby("segment").agg(
    customers=("customer_id", "count"),
    revenue=("monetary", "sum"),
    avg_recency=("recency", "mean"),
    churn_rate=("churn", "mean"),
)
summary["pct_customers"] = summary["customers"] / summary["customers"].sum() * 100
summary["pct_revenue"] = summary["revenue"] / summary["revenue"].sum() * 100
print(summary.round(1).sort_values("revenue", ascending=False))
print("Churn rate keseluruhan:", round(rfm["churn"].mean() * 100, 1), "%")

                 customers     revenue  avg_recency  churn_rate  \
segment                                                           
Champions             1291  11836647.7         20.3         0.0   
Loyal                 1147   2592612.6         74.1         0.3   
At Risk                613   1502621.7        360.9         1.0   
Lost                  1732    737622.1        452.8         1.0   
Needs Attention        860    624794.6         74.4         0.3   
New                    238     80505.4         30.4         0.0   

                 pct_customers  pct_revenue  
segment                                      
Champions                 22.0         68.1  
Loyal                     19.5         14.9  
At Risk                   10.4          8.6  
Lost                      29.5          4.2  
Needs Attention           14.6          3.6  
New                        4.0          0.5  
Churn rate keseluruhan: 50.9 %


In [7]:
# bersihkan sisa merge dari percobaan sebelumnya
rfm = rfm.drop(columns=[c for c in ["country", "country_x", "country_y"] if c in rfm.columns])

# negara terakhir tiap customer
country = (valid.sort_values("invoice_date")
                .groupby("customer_id")["country"].last()
                .reset_index())
rfm = rfm.merge(country, on="customer_id", how="left")

# data transaksi untuk dashboard (hanya transaksi valid) + kolom RFM per customer
dashboard = valid[["invoice", "invoice_date", "customer_id", "country",
                   "quantity", "price", "revenue"]].copy()
dashboard["invoice_month"] = dashboard["invoice_date"].dt.to_period("M").dt.to_timestamp()
dashboard = dashboard.merge(rfm.drop(columns="country"), on="customer_id", how="left")

# daftar churn
churn_list = rfm[rfm["churn"] == 1].sort_values("monetary", ascending=False)

# cek konsistensi sebelum ekspor
print("Kolom rfm:", list(rfm.columns))
print("Baris dashboard:", len(dashboard))
print("Revenue dashboard :", round(dashboard["revenue"].sum(), 2))
print("Monetary total RFM:", round(rfm["monetary"].sum(), 2))
print("Customer churn:", len(churn_list))

# ekspor
folder = r"C:\Users\ThinkPad\Desktop\IKEA-main\IKEA Gifting Dataset" + "\\"
rfm.to_csv(folder + "rfm_customers.csv", index=False)
churn_list.to_csv(folder + "churn_customers.csv", index=False)
dashboard.to_csv(folder + "dashboard_data.csv", index=False)

Kolom rfm: ['customer_id', 'recency', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score', 'rfm_score', 'segment', 'churn', 'churn_status', 'country']
Baris dashboard: 779495
Revenue dashboard : 17374804.27
Monetary total RFM: 17374804.27
Customer churn: 2991
